# 历史查询数据库的构建(在 train 段)

---
## 1 Imports and load splits

In [36]:
import numpy as np
import pandas as pd
import pickle
import json
from pathlib import Path

DATA_DIR = Path("data")

with open(DATA_DIR / "splits.json") as f:
    splits = json.load(f)

train_start, train_end = pd.Timestamp(splits["train"]['start']), pd.Timestamp(splits["train"]['end'])
val_start, val_end     = pd.Timestamp(splits["val"]['start']),   pd.Timestamp(splits["val"]['end'])
test_start, test_end   = pd.Timestamp(splits["test"]['start']),  pd.Timestamp(splits["test"]['end'])

---
## 2  Rebuild daily_close and r_next from minute data (unified source)

In [37]:
minute_df = pd.read_parquet(DATA_DIR / "spy_minute_clean.parquet")

daily_close = minute_df["close"].resample("D").last().dropna()
daily_close.index = daily_close.index.normalize()
daily_close.name = "close"

log_close = np.log(daily_close)
r_next = log_close.shift(-1) - log_close
r_next.name = "r_next"

print("daily_close range:", daily_close.index.min().date(), "->", daily_close.index.max().date())
print("daily_close n   :", len(daily_close))
print("r_next non-NaN n:", r_next.notna().sum())


daily_close range: 2015-01-02 -> 2025-12-31
daily_close n   : 2745
r_next non-NaN n: 2744


---
## 3 Load P model bundle and existing train signal

In [38]:
with open(DATA_DIR / "P_model.pkl", "rb") as f:
    P_bundle = pickle.load(f)

# 适配 regime-conditional 模型结构
regime_models   = P_bundle["regime_models"]    # {0: Ridge, 1: Ridge}
scaler_resid    = P_bundle["scaler_resid"]
feature_names   = P_bundle["feature_names"]
alpha_locked    = P_bundle["alpha_locked"]     # {0: 1.428, 1: 0.7848}
REGIME_LABELS   = P_bundle["regime_labels"]
K_FINAL         = P_bundle["K_FINAL"]
feature_set     = P_bundle["feature_set"]

print("K_FINAL      :", K_FINAL)
print("feature_set  :", feature_set)
print("feature_names:", feature_names)
print("alpha r0     :", alpha_locked[0])
print("alpha r1     :", alpha_locked[1])

signal_train = pd.read_parquet(DATA_DIR / "P_signal_train.parquet")["signal_regime_t"]
print("\nsignal_train range:", signal_train.index.min().date(), "->", signal_train.index.max().date())
print("signal_train n    :", len(signal_train))

K_FINAL      : 2
feature_set  : residual_only_4dim
feature_names: ['CloseLoc_resid', 'SignedVolume_z252_resid', 'PriceVolCorr_z252_resid', 'Overnight_resid', 'Mom_60d_resid', 'Mom_20d_resid']
alpha r0     : 1.4384
alpha r1     : 0.0695

signal_train range: 2017-01-12 -> 2023-04-18
signal_train n    : 1559


---
## 4 Generate signal for val and test segments

In [39]:
X_P_residual = pd.read_parquet(DATA_DIR / "X_P_residual.parquet")
regime_all   = pd.read_parquet(DATA_DIR / "regime_forward.parquet")["regime"]

print("X_P_residual columns:", list(X_P_residual.columns))
print("X_P_residual range  :", X_P_residual.index.min().date(), "->", X_P_residual.index.max().date())

base_cols = [c.replace("_resid", "") for c in feature_names]
print("\nfeature_names (model)   :", feature_names)
print("base_cols (in residual) :", base_cols)

BOUNDARY_DROP = 5

def generate_signal(seg_start, seg_end, drop_head=BOUNDARY_DROP, drop_tail=BOUNDARY_DROP):
    seg = X_P_residual.loc[seg_start:seg_end, base_cols].copy()
    seg = seg.dropna()
    if drop_head > 0:
        seg = seg.iloc[drop_head:]
    if drop_tail > 0:
        seg = seg.iloc[:-drop_tail]

    # 对齐 regime 标签
    seg_regime = regime_all.reindex(seg.index)

    X_scaled = scaler_resid.transform(seg.values)

    # 按 regime 分别预测，拼回完整时序
    pred = np.full(len(seg), np.nan)
    for r in range(K_FINAL):
        mask = (seg_regime.values == r)
        if mask.sum() == 0:
            continue
        pred[mask] = regime_models[r].predict(X_scaled[mask])

    return pd.Series(pred, index=seg.index, name="signal_t")

signal_val  = generate_signal(val_start,  val_end)
signal_test = generate_signal(test_start, test_end)

print("\nsignal_val  range:", signal_val.index.min().date(),  "->", signal_val.index.max().date(),  "| n =", len(signal_val))
print("signal_test range:", signal_test.index.min().date(), "->", signal_test.index.max().date(), "| n =", len(signal_test))

print("\n=== signal distributions ===")
print("train:"); print(signal_train.describe())
print("\nval  :"); print(signal_val.describe())
print("\ntest :"); print(signal_test.describe())

X_P_residual columns: ['CloseLoc', 'SignedVolume_z252', 'PriceVolCorr_z252', 'Overnight', 'Mom_60d', 'Mom_20d']
X_P_residual range  : 2017-01-05 -> 2025-12-31

feature_names (model)   : ['CloseLoc_resid', 'SignedVolume_z252_resid', 'PriceVolCorr_z252_resid', 'Overnight_resid', 'Mom_60d_resid', 'Mom_20d_resid']
base_cols (in residual) : ['CloseLoc', 'SignedVolume_z252', 'PriceVolCorr_z252', 'Overnight', 'Mom_60d', 'Mom_20d']

signal_val  range: 2023-05-03 -> 2024-08-20 | n = 326
signal_test range: 2024-09-05 -> 2025-12-23 | n = 327

=== signal distributions ===
train:
count    1559.000000
mean        0.000387
std         0.002403
min        -0.012769
25%        -0.000558
50%         0.000443
75%         0.001286
max         0.016191
Name: signal_regime_t, dtype: float64

val  :
count    326.000000
mean       0.000860
std        0.001377
min       -0.003948
25%       -0.000024
50%        0.000906
75%        0.001697
max        0.005958
Name: signal_t, dtype: float64

test :
count    327.

---
## 5 Inspect regime files schema

In [40]:
regime_forward   = pd.read_parquet(DATA_DIR / "regime_forward.parquet")
regime_posterior = pd.read_parquet(DATA_DIR / "regime_posterior.parquet")

print("=== regime_forward ===")
print("columns:", list(regime_forward.columns))
print("dtype:")
print(regime_forward.dtypes)
print("range :", regime_forward.index.min().date(), "->", regime_forward.index.max().date())
print("head:")
print(regime_forward.head(3))

print("\n=== regime_posterior ===")
print("columns:", list(regime_posterior.columns))
print("dtype:")
print(regime_posterior.dtypes)
print("range :", regime_posterior.index.min().date(), "->", regime_posterior.index.max().date())
print("head:")
print(regime_posterior.head(3))

=== regime_forward ===
columns: ['regime']
dtype:
regime    int64
dtype: object
range : 2015-06-25 -> 2025-12-31
head:
            regime
date              
2015-06-25       1
2015-06-26       1
2015-06-29       0

=== regime_posterior ===
columns: ['p_r0', 'p_r1']
dtype:
p_r0    float64
p_r1    float64
dtype: object
range : 2015-06-25 -> 2025-12-31
head:
                    p_r0          p_r1
date                                  
2015-06-25  2.155101e-59  1.000000e+00
2015-06-26  1.822264e-03  9.981777e-01
2015-06-29  1.000000e+00  6.125271e-13


---
## 6 Build train triples

In [41]:
signal_all = pd.concat([signal_train, signal_val, signal_test])
signal_all.name = "signal"

triples_train = pd.DataFrame({
    "signal":       signal_train,
    "regime":       regime_forward["regime"],
    "p_r0":         regime_posterior["p_r0"],
    "p_r1":         regime_posterior["p_r1"],
    "r_next":       r_next,
}).dropna(subset=["signal", "regime", "r_next"])

triples_train = triples_train.loc[
    (triples_train.index >= signal_train.index.min()) &
    (triples_train.index <= signal_train.index.max())
]

print("train triples shape:", triples_train.shape)
print("date range         :", triples_train.index.min().date(), "->", triples_train.index.max().date())
print("\nregime distribution in train triples:")
print(triples_train["regime"].value_counts().sort_index())

print("\nany NaN in critical fields:")
print(triples_train[["signal", "regime", "r_next"]].isna().sum())

print("\nhead:")
print(triples_train.head(3))
print("\ntail:")
print(triples_train.tail(3))

train triples shape: (1559, 5)
date range         : 2017-01-12 -> 2023-04-18

regime distribution in train triples:
regime
0.0    737
1.0    822
Name: count, dtype: int64

any NaN in critical fields:
signal    0
regime    0
r_next    0
dtype: int64

head:
              signal  regime      p_r0      p_r1    r_next
2017-01-12  0.001283     1.0  0.000143  0.999857  0.002028
2017-01-13  0.000747     1.0  0.000068  0.999932 -0.003265
2017-01-17  0.001669     1.0  0.000135  0.999865  0.001987

tail:
              signal  regime      p_r0      p_r1    r_next
2023-04-14  0.000805     1.0  0.000161  0.999839  0.003509
2023-04-17  0.000378     1.0  0.000176  0.999824  0.000676
2023-04-18  0.000020     1.0  0.000892  0.999108 -0.000024


---
## 7 Compute regime-specific signal bin edges (quantile-based)

In [42]:
BIN_QUANTILES = {
    0: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    1: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
}

bin_edges = {}
for r, qs in BIN_QUANTILES.items():
    sub = triples_train.loc[triples_train["regime"] == r, "signal"]
    edges = sub.quantile(qs).values
    edges[0]  = -np.inf
    edges[-1] = +np.inf
    bin_edges[r] = edges.tolist()

print("=== Bin edges per regime ===")
for r, edges in bin_edges.items():
    print(f"r{r} ({len(edges)-1} bins): {[f'{e:+.6f}' for e in edges]}")

print("\nstrictly increasing check:")
for r, edges in bin_edges.items():
    interior = [e for e in edges if np.isfinite(e)]
    is_inc = all(interior[i] < interior[i+1] for i in range(len(interior)-1))
    print(f"  r{r}: {is_inc}")

=== Bin edges per regime ===
r0 (5 bins): ['-inf', '-0.002405', '-0.000581', '+0.000981', '+0.003065', '+inf']
r1 (5 bins): ['-inf', '-0.000129', '+0.000308', '+0.000691', '+0.001124', '+inf']

strictly increasing check:
  r0: True
  r1: True


---
## 8 Assign signal_bin to each train triple and check cell counts

In [43]:
def assign_bin(row, edges_dict):
    edges = edges_dict[int(row["regime"])]
    return int(np.digitize(row["signal"], edges[1:-1]))

triples_train["signal_bin"] = triples_train.apply(lambda r: assign_bin(r, bin_edges), axis=1)

cell_counts = triples_train.groupby(["regime", "signal_bin"]).size().rename("n").reset_index()
print("=== Cell counts (regime, signal_bin) -> n ===")
print(cell_counts.to_string(index=False))



=== Cell counts (regime, signal_bin) -> n ===
 regime  signal_bin   n
    0.0           0 148
    0.0           1 147
    0.0           2 147
    0.0           3 147
    0.0           4 148
    1.0           0 165
    1.0           1 164
    1.0           2 164
    1.0           3 164
    1.0           4 165


---
## 9 Compute basic per-cell statistics

In [44]:
def cell_stats(group):
    r = group["r_next"].values
    n = len(r)
    return pd.Series({
        "n":          n,
        "mean_ret":   r.mean(),
        "median_ret": np.median(r),
        "std_ret":    r.std(ddof=1),
        "win_rate":   (r > 0).mean(),
        "sharpe_ann": r.mean() / r.std(ddof=1) * np.sqrt(252) if r.std(ddof=1) > 0 else np.nan,
    })

lookup = triples_train.groupby(["regime", "signal_bin"]).apply(cell_stats).reset_index()
lookup["regime"] = lookup["regime"].astype(int)
lookup["signal_bin"] = lookup["signal_bin"].astype(int)
lookup["n"] = lookup["n"].astype(int)

print("=== Lookup table (no CI yet) ===")
print(lookup.to_string(index=False, float_format=lambda x: f"{x:+.6f}"))

=== Lookup table (no CI yet) ===
 regime  signal_bin   n  mean_ret  median_ret   std_ret  win_rate  sharpe_ann
      0           0 148 -0.004162   -0.001257 +0.015273 +0.425676   -4.325515
      0           1 147 -0.000163   +0.000885 +0.012327 +0.510204   -0.209582
      0           2 147 +0.000111   +0.000329 +0.014457 +0.523810   +0.121729
      0           3 147 +0.000414   +0.001629 +0.013053 +0.544218   +0.503873
      0           4 148 +0.005228   +0.004994 +0.018610 +0.601351   +4.459779
      1           0 165 -0.000833   -0.000282 +0.008529 +0.484848   -1.551281
      1           1 164 +0.000437   +0.000454 +0.005075 +0.567073   +1.367666
      1           2 164 +0.001107   +0.000818 +0.005592 +0.597561   +3.142454
      1           3 164 +0.000564   +0.000989 +0.006571 +0.591463   +1.361290
      1           4 165 +0.001115   +0.001433 +0.007456 +0.600000   +2.374665


C:\Users\Bobli\AppData\Local\Temp\ipykernel_51776\531997157.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lookup = triples_train.groupby(["regime", "signal_bin"]).apply(cell_stats).reset_index()


---
## 10 Block bootstrap CI for mean_ret (block length = 10, B = 1000)

In [45]:
RNG = np.random.default_rng(42)
BLOCK_LEN = 10
N_BOOT = 1000

def block_bootstrap_ci(returns, block_len=BLOCK_LEN, n_boot=N_BOOT, alpha=0.05, rng=RNG):
    n = len(returns)
    if n < block_len:
        return np.nan, np.nan
    n_blocks = int(np.ceil(n / block_len))
    means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, n - block_len + 1, size=n_blocks)
        sample = np.concatenate([returns[s:s + block_len] for s in starts])[:n]
        means[b] = sample.mean()
    lo = np.quantile(means, alpha / 2)
    hi = np.quantile(means, 1 - alpha / 2)
    return lo, hi

ci_rows = []
for (r, b), grp in triples_train.sort_index().groupby(["regime", "signal_bin"]):
    rets = grp["r_next"].values
    lo, hi = block_bootstrap_ci(rets)
    ci_rows.append({"regime": int(r), "signal_bin": int(b), "ci95_low": lo, "ci95_high": hi})

ci_df = pd.DataFrame(ci_rows)
lookup = lookup.merge(ci_df, on=["regime", "signal_bin"])

def confidence_label(row):
    if row["n"] < 30:
        return "no_trade"
    if row["ci95_low"] <= 0 <= row["ci95_high"]:
        return "low"
    return "high"

lookup["confidence"] = lookup.apply(confidence_label, axis=1)

print("=== Lookup table with CI and confidence ===")
print(lookup.to_string(index=False, float_format=lambda x: f"{x:+.6f}"))

print("\n=== Confidence label counts ===")
print(lookup["confidence"].value_counts())

=== Lookup table with CI and confidence ===
 regime  signal_bin   n  mean_ret  median_ret   std_ret  win_rate  sharpe_ann  ci95_low  ci95_high confidence
      0           0 148 -0.004162   -0.001257 +0.015273 +0.425676   -4.325515 -0.006947  -0.001767       high
      0           1 147 -0.000163   +0.000885 +0.012327 +0.510204   -0.209582 -0.002213  +0.001260        low
      0           2 147 +0.000111   +0.000329 +0.014457 +0.523810   +0.121729 -0.001684  +0.001999        low
      0           3 147 +0.000414   +0.001629 +0.013053 +0.544218   +0.503873 -0.002145  +0.002827        low
      0           4 148 +0.005228   +0.004994 +0.018610 +0.601351   +4.459779 +0.003205  +0.007161       high
      1           0 165 -0.000833   -0.000282 +0.008529 +0.484848   -1.551281 -0.002457  +0.000432        low
      1           1 164 +0.000437   +0.000454 +0.005075 +0.567073   +1.367666 -0.000264  +0.001050        low
      1           2 164 +0.001107   +0.000818 +0.005592 +0.597561   +3.14245

--
## 12 save output

In [46]:
triples_train.to_parquet(DATA_DIR / "train_triples.parquet")

bin_edges_serializable = {
    str(r): [float(e) if np.isfinite(e) else ("-inf" if e < 0 else "inf") for e in edges]
    for r, edges in bin_edges.items()
}
with open(DATA_DIR / "signal_bins.json", "w") as f:
    json.dump({
        "bin_quantiles": {str(k): v for k, v in BIN_QUANTILES.items()},
        "bin_edges":     bin_edges_serializable,
        "note":          "edges[0]/edges[-1] stored as 'inf'/'-inf' strings; reconstruct with np.inf",
    }, f, indent=2)

lookup.to_parquet(DATA_DIR / "lookup_table.parquet")

print("Saved:")
print("  data/train_triples.parquet :", triples_train.shape)
print("  data/signal_bins.json")
print("  data/lookup_table.parquet  :", lookup.shape)

Saved:
  data/train_triples.parquet : (1559, 6)
  data/signal_bins.json
  data/lookup_table.parquet  : (10, 11)
